In [17]:
!pip install -U gpt4all psutil scikit-learn pandas safetensors \
transformers peft accelerate huggingface_hub

  Using cached gpt4all-2.8.2-py3-none-win_amd64.whl.metadata (4.8 kB)
  Using cached psutil-7.1.0-cp37-abi3-win_amd64.whl.metadata (23 kB)
  Using cached scikit_learn-1.7.2-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
Using cached gpt4all-2.8.2-py3-none-win_amd64.whl (119.6 MB)
Using cached psutil-7.1.0-cp37-abi3-win_amd64.whl (247 kB)
Using cached scikit_learn-1.7.2-cp312-cp312-win_amd64.whl (8.7 MB)
Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl (11.0 MB)
Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl (320 kB)
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.5.3
    Uninstalling safetensors-0.5.3:
      Successfully uninstalled safetensors-0.5.3
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.0
    Uninstalling psutil-5.9.0:
      Successfully uninstalled psutil-5.9.0
 

  You can safely remove it manually.


In [51]:
!pip install -U gguf sentencepiece

  Using cached gguf-0.17.1-py3-none-any.whl.metadata (4.3 kB)
Using cached gguf-0.17.1-py3-none-any.whl (96 kB)


In [53]:
from pathlib import Path
import os

# Use script folder if available; else current working dir (for notebooks)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd().resolve()

DATA_PATH = HERE / "test_dataset.json"  # your JSON file next to the script

# --- Robust resolver for converter and quantizer filenames ---
def first_exists(candidates):
    for name in candidates:
        p = HERE / name
        if p.exists():
            return p
    return None

# Converter can be named with dashes or underscores, with or without .py
CONVERTER = first_exists([
    "convert-hf-to-gguf.py",
    "convert_hf_to_gguf.py",
    "convert-hf-to-gguf",     # if extensions are hidden
    "convert_hf_to_gguf"
])
if CONVERTER is None:
    raise FileNotFoundError(
        f"Couldn't find converter next to the script. Put one of these names in {HERE}:\n"
        f"  convert-hf-to-gguf.py  OR  convert_hf_to_gguf.py"
    )

# Quantizer may be 'quantize.exe' or 'llama-quantize.exe' (from Windows zip)
QUANT = first_exists([
    "quantize.exe",
    "llama-quantize.exe",
    "quantize",
    "llama-quantize"
])
if QUANT is None:
    raise FileNotFoundError(
        f"Couldn't find quantizer next to the script. Put one of these names in {HERE}:\n"
        f"  quantize.exe  OR  llama-quantize.exe"
    )

# Artifacts
ARTI = HERE / "artifacts"
MERGED = ARTI / "merged_fp16"
GGUF_F16 = ARTI / "model-f16.gguf"
GGUF_Q4  = ARTI / "model.Q4_K_M.gguf"

print("Using folder:", HERE)
print("Dataset   :", DATA_PATH)
print("Converter :", CONVERTER.name)
print("Quantizer :", QUANT.name)

Using folder: D:\Master\__Masterprojekt\git-application\llm-finetuning
Dataset   : D:\Master\__Masterprojekt\git-application\llm-finetuning\test_dataset.json
Converter : convert_hf_to_gguf.py
Quantizer : llama-quantize.exe


In [55]:
# ========= CONFIG (edit these 4 lines) =========
BASE_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"      # your base model on HF
LORA_ID   = "eduhuemar001/tinyllama-german-sentiment-4bit"               # your LoRA adapters on HF
# ===============================================

import os, sys, time, subprocess, psutil, pandas as pd, torch
from pathlib import Path
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from gpt4all import GPT4All

def run(cmd):
    print("\n[cmd]", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

def need(p: Path, name: str):
    if not p.exists():
        raise FileNotFoundError(f"{name} not found at: {p}")

def load_dataset(p: Path) -> pd.DataFrame:
    need(p, "dataset")
    # Prefer JSON Lines; fall back to array
    try:
        df = pd.read_json(p, lines=True)
    except ValueError:
        df = pd.read_json(p, lines=False)
    need_cols = {"review_text","sentiment"}
    if not need_cols.issubset(df.columns):
        raise ValueError(f"Dataset must contain {need_cols}, got {set(df.columns)}")
    return df

def merge_lora_to_fp16():
    print("[step] Merge LoRA → base (CPU, fp16)…")
    tok = AutoTokenizer.from_pretrained(BASE_ID, use_fast=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.float16, device_map={"": "cpu"})
    merged = PeftModel.from_pretrained(base, LORA_ID, device_map={"": "cpu"}).merge_and_unload()
    MERGED.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(MERGED, safe_serialization=True)
    tok.save_pretrained(MERGED)
    print("[ok] saved merged HF model to", MERGED)

def convert_to_gguf():
    need(CONVERTER, "convert_hf_to_gguf.py")
    ARTI.mkdir(parents=True, exist_ok=True)
    print("[step] Convert HF → GGUF (fp16)…")
    run([sys.executable, str(CONVERTER), str(MERGED), "--outfile", str(GGUF_F16)])
    need(GGUF_F16, "fp16 GGUF")
    print("[ok] wrote", GGUF_F16)

def quantize_q4km():
    need(QUANT, "quantize(.exe)")
    print("[step] Quantize to 4-bit Q4_K_M…")
    run([str(QUANT), str(GGUF_F16), str(GGUF_Q4), "Q4_K_M"])
    need(GGUF_Q4, "Q4_K_M GGUF")
    print("[ok] wrote", GGUF_Q4)

def evaluate_gguf():
    print("[step] Load Q4 GGUF on CPU (in-process, no server)…")
    llm = GPT4All(model=str(GGUF_Q4), model_path=str(GGUF_Q4.parent), allow_download=False, verbose=False)
    # Optional: set thread count (comment out if you prefer defaults)
    try:
        llm.model.backend.thread_count = max(1, (os.cpu_count() or 2)//2)
    except Exception:
        pass

    df = load_dataset(DATA_PATH)
    instr = ("### Instruction:\n"
             "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
             "### Bewertung:\n")
    ansp  = "\n\n### Antwort:\n"
    valid = {"positive","neutral","negative"}

    preds, trues = [], []
    proc = psutil.Process(os.getpid()); m0 = proc.memory_info().rss; t0 = time.time()

    for _, row in df.iterrows():
        prompt = instr + str(row["review_text"]) + ansp
        txt = llm.generate(prompt, max_tokens=2, temp=0.0, top_p=1.0, top_k=0,
                           stopping_strings=["\n","\r"]).strip().lower()
        lab = (txt.split() or [""])[0]
        if lab not in valid and "### antwort:" in txt:
            lab = txt.split("### antwort:")[-1].strip().split()[0]
        if lab not in valid:
            lab = "neutral"
        preds.append(lab)
        trues.append(str(row["sentiment"]).strip().lower())

    wall = time.time()-t0
    dmb  = (proc.memory_info().rss - m0)/1024**2
    print(f"\nInference wallclock: {wall:.2f}s   ΔRAM(proc): {dmb:.1f} MB")
    print("\nClassification report:")
    print(classification_report(trues, preds, digits=3))

if __name__ == "__main__":
    # Only do the heavy steps if artifacts don’t exist yet
    if not MERGED.exists():      merge_lora_to_fp16()
    if not GGUF_F16.exists():    convert_to_gguf()
    if not GGUF_Q4.exists():     quantize_q4km()
    evaluate_gguf()

[step] Convert HF → GGUF (fp16)…

[cmd] C:\Users\marku\anaconda3\python.exe D:\Master\__Masterprojekt\git-application\llm-finetuning\convert_hf_to_gguf.py D:\Master\__Masterprojekt\git-application\llm-finetuning\artifacts\merged_fp16 --outfile D:\Master\__Masterprojekt\git-application\llm-finetuning\artifacts\model-f16.gguf


CalledProcessError: Command '['C:\\Users\\marku\\anaconda3\\python.exe', 'D:\\Master\\__Masterprojekt\\git-application\\llm-finetuning\\convert_hf_to_gguf.py', 'D:\\Master\\__Masterprojekt\\git-application\\llm-finetuning\\artifacts\\merged_fp16', '--outfile', 'D:\\Master\\__Masterprojekt\\git-application\\llm-finetuning\\artifacts\\model-f16.gguf']' returned non-zero exit status 1.